### 멀티모달 모델
기존의 LLM이 텍스트라는 단일 모드만 처리했다면, 멀티모달 모델은 여러 가지 형태의 정보를 함께 처리할 수 있다.

1. CLIP (Contrastive Language-Image Pre-training)
- OpenAI에서 발표한 모델로, 텍스트와 이미지를 하나의 공간에서 연결하는 혁신적인 방법론이다.
- 기존 AI가 이미지를 "강아지", "고양이"라는 정해진 클래스로만 배웠다면, CLIP은 자연어 설명을 통해 이미지의 맥락을 배운다.

2. 공동 임베딩 공간(Shared Embedding Space)
- 멀티모달 모델은 텍스트와 이미지를 각각의 특징 벡터로 변환한 뒤, 이를 하나의 좌표 평면에 배치한다.
- 텍스트 인코더: "해변을 달리는 골든 리트리버"라는 문장을 벡터로 변환.
- 이미지 인코더: 실제 강아지 사진을 벡터로 변환.
- 대조 학습(Contrastive Learning): 관련 있는 텍스트와 이미지 벡터는 가까워지도록 유도하고, 관련 없는 데이터끼리는 멀어지도록 학습한다.

`관련 문서: https://contents.premium.naver.com/byline/commercebn/contents/260421165257027nn`

3. 벡터 유사도 (Vector Similarity)
- AI는 두 데이터가 얼마나 비슷한지 수학적으로 계산하며, 주로 코사인 유사도(Cosine Similarity)를 사용한다.
- 유사도 1에 가까움: 이 텍스트는 이미지의 정확한 설명이다.
- 유사도 0에 가까움: 이 텍스트와 이미지는 아무 상관이 없다.

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

In [7]:
import os
import base64
from langchain_core.messages import HumanMessage

class SimpleMultiModal:
    def __init__(self, llm):
        self.llm = llm

    def stream(self, input_source, prompt="이 이미지를 설명해줘"):
        # 로컬 파일인지 확인
        if os.path.exists(input_source):
            with open(input_source, 'rb') as f:
                data = base64.b64encode(f.read()).decode("utf-8")
                image_url = f'data:image/jpeg;base64,{data}'
        else:
            image_url = input_source

        message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}}
        ])

        return self.llm.stream([message])        

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano"
)

smm = SimpleMultiModal(llm)
stream = smm.stream('./images/톰.jpg', prompt="고양이와 강아지 중 어디에 더 가까운가?")
stream_response(stream)

사진 속 고양이는 **고양이 쪽에 더 가깝습니다.** 😺  
(귀 모양, 얼굴 형태, 수염/주둥이 비율 등 전형적인 고양이 특징이 뚜렷해요.)

In [9]:
import os
import base64
from langchain_core.messages import HumanMessage, SystemMessage

class PersonalMultiModal:
    def __init__(self, llm, system_prompt=None, human_prompt="이 이미지를 설명해줘"):
        self.llm = llm
        self.system_prompt = system_prompt
        self.human_prompt = human_prompt

    def _encode_image(self, image_path):
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")

    def stream(self, input_source):
        if os.path.exists(input_source):
            data = self._encode_image(input_source)
            image_url = f"data:image/jpeg;base64,{data}"
        else:
            image_url = input_source

        messages = []
        
        if self.system_prompt:
            messages.append(SystemMessage(content=self.system_prompt))
            
        messages.append(
            HumanMessage(content=[
                {"type": "text", "text": self.human_prompt},
                {"type": "image_url", "image_url": {"url": image_url}}
            ])
        )

        return self.llm.stream(messages)

In [10]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano"
)

system_prompt = """
당신은 매출표를 분석하는 친절한 경영학 박사 AI입니다.
답변 이후 사용자에게 추가적인 질문은 하지 않습니다.
"""

human_prompt = """
주어진 표는 회사의 매출표 입니다. 
당기 순이익 및 향후 인사이트를 제공하세요.
5문장 이내로 답변해줘.
"""

pmm = PersonalMultiModal(llm, system_prompt, human_prompt)

stream = pmm.stream("./images/매출표.png")
stream_response(stream)

아래 표를 기준으로 **당기 순이익**을 먼저 산출하고, 그 다음 **향후 인사이트(수익성/구조/리스크 관점)**를 정리하겠습니다.

---

## 1) 당기 순이익(당기 순이익 및 계산 근거)

표의 흐름을 그대로 따라가면:

- **영업이익**: 230,000,000  
- **영업외수익**: 15,000,000  
- **영업외비용**: -35,000,000  
- **법인세차감전이익**: 210,000,000  *(= 230,000,000 + 15,000,000 - 35,000,000)*  
- **법인세비용**: -42,000,000 *(법인세율 20%)*  
- **당기 순이익**: **168,000,000**

✅ **당기 순이익 = 210,000,000 × (1 - 20%) = 168,000,000**

---

## 2) 핵심 인사이트(향후 관점)

### (1) 매출총이익률은 양호하나, “영업비용”이 이익을 깎는 구조
- 매출액: 1,500,000,000  
- 매출원가: 800,000,000  
- **매출총이익(=700,000,000)**  
- **매출총이익률 = 700,000,000 / 1,500,000,000 = 46.7%**

즉, **본업의 원가 구조는 상당히 좋습니다.**  
그런데 영업이익이 230,000,000으로 줄어드는 이유는 **판매관리비(영업비용) 320,000,000**이 크기 때문입니다.

- 판매관리비 비중: 320,000,000 / 1,500,000,000 = **21.3%**

➡️ **향후 우선순위는 “매출총이익률 방어” + “판매관리비 효율화”**입니다.

---

### (2) 영업외는 순손실(비용 우위) → 재무/비영업 요인 관리 필요
- 영업외수익: +15,000,000  
- 영업외비용: -35,000,000  
- **영업외 순손실 = -20,000,000**

➡️ 본업(영업이익)은 플러스인데, **비영업 영역에서 2천만 원이 추가로 깎이고** 있습니다.  
향후에는:
- 이자비용/기타비용(표의